# Feature Engineering

This notebook creates derived features and aggregated metrics from the cleaned dataset. These engineered features provide meaningful measures of trader performance and behavior that will be used during exploratory data analysis.

## Objectives

The objectives of this notebook are to:

- Load the cleaned and merged dataset produced during the data preparation stage.
- Create new features that better represent trader performance and trading behavior.
- Generate aggregated metrics at the daily and trader levels.
- Prepare an enriched dataset for exploratory data analysis and visualization.

In [1]:
import pandas as pd

merged_df = pd.read_csv(
    "../data/processed/trader_sentiment_processed.csv",
    parse_dates=["Timestamp IST", "date"]
)
merged_df.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,timestamp,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed


## Daily Profit and Loss per Trader

The total daily profit and loss (PnL) is calculated for each trader by aggregating the `Closed PnL` values for every account on each trading day.

This metric helps evaluate trader performance over time and enables comparison of profitability under different market sentiment conditions.

In [2]:
# Daily PnL per trader
daily_pnl = (
    merged_df
    .groupby(["date", "Account"])["Closed PnL"]
    .sum()
    .reset_index()
)
daily_pnl.head()

,date,Account,Closed PnL
0,2023-05-01,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,0.000000
1,2023-12-05,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,0.000000
2,2023-12-14,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,-205.434737
3,2023-12-15,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,-24.632034
4,2023-12-16,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,0.000000


## Win/Loss Indicator

A binary win/loss indicator is created based on the `Closed PnL` column to classify each trade as either profitable or non-profitable.

- **Win (1):** Trade closed with a positive profit (`Closed PnL > 0`)
- **Loss (0):** Trade closed with zero or negative profit (`Closed PnL ≤ 0`)

This feature simplifies the calculation of win rates across different traders and market sentiment conditions.

In [3]:
# Create a column indicating whether a trade was profitable
merged_df["Win"] = (merged_df["Closed PnL"] > 0).astype(int)

## Win Rate per Trader

The win rate per trader is calculated to measure the consistency of each trader's performance. A **Win** is defined as any trade with a positive realized profit (`Closed PnL > 0`).

For each trader (identified by the `Account` column), the win rate is computed as:

- **Number of profitable trades**
- divided by
- **Total number of trades executed**

In [4]:
# Calculate win rate per trader
win_rate_per_trader = (
    merged_df
    .groupby("Account")
    .agg(
        Total_Trades=("Win", "count"),
        Winning_Trades=("Win", "sum"),
        Win_Rate=("Win", "mean")
    )
    .reset_index()
)

# Convert to percentage
win_rate_per_trader["Win_Rate"] *= 100

win_rate_per_trader.head()

,Account,Total_Trades,Winning_Trades,Win_Rate
0,0x083384f897ee0f19899168e3b1bec365f52a9012,3818,1373,35.961236
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,7280,3223,44.271978
2,0x271b280974205ca63b716753467d5a371de622ab,3809,1150,30.191651
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,13311,5838,43.858463
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3239,1684,51.991355


## Trade Count per Trader

The trade count per trader is calculated by counting the total number of trades executed by each trader (identified by the `Account` column).

This feature measures the trading activity of individual traders and serves as an indicator of their participation level in the market. It will later be used to segment traders into groups such as frequent and infrequent traders during the exploratory data analysis stage.

In [5]:
trade_count = (
    merged_df
    .groupby("classification")
    .size()
    .reset_index(name="Trades")
)

In [6]:
trade_count_per_trader = (
    merged_df
    .groupby("Account")
    .size()
    .reset_index(name="Trade Count")
)
trade_count_per_trader.head()

,Account,Trade Count
0,0x083384f897ee0f19899168e3b1bec365f52a9012,3818
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,7280
2,0x271b280974205ca63b716753467d5a371de622ab,3809
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,13311
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3239


## Average Trade Size per Trader

The average trade size per trader is calculated by computing the mean trade value (`Size USD`) for each trader (identified by the `Account` column).

This feature represents the average amount of capital deployed by a trader in each trade and serves as an indicator of their trading style and risk exposure. Traders with higher average trade sizes may exhibit greater risk tolerance compared to those executing smaller trades.

In [7]:
average_trade_size_per_trader = (
    merged_df
    .groupby("Account")["Size USD"]
    .mean()
    .reset_index(name="Average Trade Size (USD)")
)

average_trade_size_per_trader.head()

,Account,Average Trade Size (USD)
0,0x083384f897ee0f19899168e3b1bec365f52a9012,16159.576734
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,1653.226327
2,0x271b280974205ca63b716753467d5a371de622ab,8893.000898
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,507.626933
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3138.894782


## Number of Trades per Day

The number of trades executed each day is calculated by counting all transactions occurring on the same date.

This feature provides insight into changes in trading activity and helps identify periods of increased or decreased market participation.

In [8]:
trades_per_day = (
    merged_df
    .groupby("date")
    .size()
    .reset_index(name="Trades")
)

## Open Positions

The trading dataset contains multiple transaction types, including opening positions, closing positions, spot trades, settlements, and other system-generated events.

For the analysis of trader positioning, only transactions corresponding to **Open Long** and **Open Short** are retained. These records represent the initiation of new market positions and are used to analyze directional trading behavior.

Creating a dedicated dataset of open positions simplifies the computation of long/short ratios and enables comparison of directional bias across different market sentiment conditions during the exploratory analysis stage.

In [9]:
open_positions = merged_df[
    merged_df["Direction"].isin(["Open Long", "Open Short"])
].copy()

open_positions.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Order ID,Crossed,Fee,Trade ID,Timestamp,date,timestamp,value,classification,Win
64,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,AAVE,235.03,7.47,1755.67,BUY,2024-12-03 14:42:00,0.00,Open Long,0.0,...,52201279961,True,0.614485,7.750000e+14,1.730000e+12,2024-12-03,1.733204e+09,76.0,Extreme Greed,0
65,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,AAVE,235.04,9.02,2120.06,BUY,2024-12-03 14:42:00,7.47,Open Long,0.0,...,52201279961,True,0.742021,5.420000e+14,1.730000e+12,2024-12-03,1.733204e+09,76.0,Extreme Greed,0
66,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,AAVE,235.04,7.72,1814.51,BUY,2024-12-03 14:42:00,16.49,Open Long,0.0,...,52201279961,True,0.635078,6.300000e+14,1.730000e+12,2024-12-03,1.733204e+09,76.0,Extreme Greed,0
67,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,AAVE,235.09,3.66,860.43,BUY,2024-12-03 14:42:00,24.21,Open Long,0.0,...,52201279961,True,0.301150,8.490000e+14,1.730000e+12,2024-12-03,1.733204e+09,76.0,Extreme Greed,0
68,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,AAVE,235.09,15.45,3632.14,BUY,2024-12-03 14:42:00,27.87,Open Long,0.0,...,52201279961,True,1.271249,7.960000e+14,1.730000e+12,2024-12-03,1.733204e+09,76.0,Extreme Greed,0


In [10]:
open_positions["Direction"].value_counts()

Direction
Open Long     49895
Open Short    39741
Name: count, dtype: int64

## Daily Trading Volume

Daily trading volume is calculated by summing the total USD value of all trades executed on each trading day.

This metric measures overall market activity and capital flow over time. It can be used to identify periods of increased or decreased trading participation and to compare trading activity under different market sentiment conditions.

In [11]:
daily_volume = (
    merged_df
    .groupby("date")["Size USD"]
    .sum()
    .reset_index()
)

## Average Daily Profit and Loss (PnL)

The average daily profit and loss is calculated by computing the mean `Closed PnL` of all trades executed on each trading day.

This metric provides an estimate of the average profitability of trades on a given day and helps identify whether traders generally performed better during Fear or Greed market conditions.

In [12]:
daily_avg_pnl = (
    merged_df
    .groupby("date")["Closed PnL"]
    .mean()
    .reset_index()
)

## Total Daily Profit and Loss (PnL)

The total daily profit and loss is calculated by summing the `Closed PnL` of all trades executed on each trading day.

This feature represents the overall trading performance for each day and can be used to examine trends in market profitability and compare aggregate trader performance across different sentiment periods.

In [13]:
daily_total_pnl = (
    merged_df
    .groupby("date")["Closed PnL"]
    .sum()
    .reset_index()
)

## Save the Engineered Dataset

After creating all engineered features, the enriched dataset is saved for use in the Exploratory Data Analysis (EDA) notebook.

Saving the processed data ensures a reproducible workflow and avoids repeating feature engineering in subsequent analyses.

In [14]:
merged_df.to_csv(
    "../data/processed/trader_sentiment_engineered.csv",
    index=False
)

In [15]:
daily_pnl.to_csv(
    "../data/processed/daily_pnl.csv",
    index=False
)
win_rate_per_trader.to_csv(
    "../data/processed/win_rate_per_trader.csv",
    index=False
)
average_trade_size_per_trader.to_csv(
    "../data/processed/average_trade_size_per_trader.csv",
    index=False
)
trades_per_day.to_csv(
    "../data/processed/trades_per_day.csv",
    index=False
)
trade_count.to_csv(
    "../data/processed/trade_count.csv",
    index=False
)
trade_count_per_trader.to_csv(
    "../data/processed/trade_count_per_trader.csv",
    index=False
)
open_positions.to_csv(
    "../data/processed/open_positions.csv",
    index=False
)
daily_volume.to_csv(
    "../data/processed/daily_trading_volume.csv",
    index=False
)
daily_avg_pnl.to_csv(
    "../data/processed/daily_average_pnl.csv",
    index=False
)
daily_total_pnl.to_csv(
    "../data/processed/daily_total_pnl.csv",
    index=False
)
print("All engineered datasets saved successfully!")

All engineered datasets saved successfully!


## Summary

In this notebook, several analytical features were engineered to better represent trader performance and trading behavior. These derived metrics, including profitability, win rate, trading activity, position bias, and sentiment-based aggregates, provide the foundation for the subsequent exploratory data analysis. The engineered dataset is now ready for visualization and statistical analysis to investigate the relationship between market sentiment and trader performance.